In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit, PredefinedSplit
from scipy.stats import uniform
import os 
print(os.getcwd())

/Users/kellianneng/Desktop/Year 4/DSE4211/btc-rv-prediction/models


In [3]:
def get_rf_params_val(Y, target_col):
    y_train = Y[target_col].values
    X_train = Y.drop(columns=[target_col])

    # 80/20 time-ordered split
    split_idx = int(len(Y) * 0.8)

    # -1 = training fold, 0 = validation fold
    test_fold = np.concatenate([
        -1 * np.ones(split_idx, dtype=int),
         0 * np.ones(len(Y) - split_idx, dtype=int)
    ])
    ps = PredefinedSplit(test_fold)

    # Tune only max_features
    # Searches between 10% and 100% of predictors
    param_dist = {
        "max_features": uniform(0.10, 0.90)
    }

    rs = RandomizedSearchCV(
        estimator=RandomForestRegressor(
            n_estimators=500,
            max_depth=None,
            bootstrap=True,
            random_state=42,
            n_jobs=-1
        ),
        param_distributions=param_dist,
        n_iter=50,
        cv=ps,
        scoring="neg_mean_squared_error",
        n_jobs=-1,
        random_state=42
    )

    rs.fit(X_train, y_train)
    return rs.best_params_


In [4]:

def get_rf_params_cv(Y, target_col):
    y_train = Y[target_col].values
    X_train = Y.drop(columns=[target_col])

    tscv = TimeSeriesSplit(n_splits=5)

    param_dist = {
        "max_features": uniform(0.10, 0.90)
    }

    rs = RandomizedSearchCV(
        estimator=RandomForestRegressor(
            n_estimators=500,
            max_depth=None,
            bootstrap=True,
            random_state=42,
            n_jobs=-1
        ),
        param_distributions=param_dist,
        n_iter=50,
        cv=tscv,
        scoring="neg_mean_squared_error",
        n_jobs=-1,
        random_state=42
    )

    rs.fit(X_train, y_train)
    return rs.best_params_


In [5]:

# ---------- MAIN ----------
train_df = pd.read_csv("../data/train_dataset.csv")
train_df["date"] = pd.to_datetime(train_df["date"], format="%d/%m/%y")
train_df = train_df.sort_values("date").reset_index(drop=True)

horizons = ["y_h1", "y_h3", "y_h5", "y_h7"]

tuning_records = []

print("Tuning RF hyperparameters (train only)...")

for h in horizons:
    others = [col for col in horizons if col != h]

    # Use only training data for tuning
    Y_init = train_df.drop(columns=others + ["date"], errors="ignore")

    # Number of predictors (excluding target)
    num_preds = Y_init.shape[1] - 1

    print(f"Tuning for {h}...")

    params_val = get_rf_params_val(Y_init, h)
    params_cv = get_rf_params_cv(Y_init, h)

    tuning_records.append({
        "horizon": h,
        "rf_val_max_features": params_val["max_features"],
        "rf_val_m_count": int(params_val["max_features"] * num_preds),
        "rf_cv_max_features": params_cv["max_features"],
        "rf_cv_m_count": int(params_cv["max_features"] * num_preds),
        "n_predictors": num_preds,
        "n_estimators": 500
    })

rf_hyperparams_df = pd.DataFrame(tuning_records)
rf_hyperparams_df.to_csv("../models/rf_hyperparameters.csv", index=False)

print("\nSaved tuned RF hyperparameters to ../models/rf_hyperparameters.csv")
print(rf_hyperparams_df)

Tuning RF hyperparameters (train only)...
Tuning for y_h1...


KeyboardInterrupt: 